# RidgeNet 5–24 kHz: training on Colab (free T4 GPU)

Trains the 5–24 kHz detector variant (`mod_detector_5_24/`) on synthetic data, exactly as `ridgenet-v2c` was trained
except for the band. The code is the project's tested package (`whistle_code_5_24.zip`); this notebook only sets up the
session and runs it.

**Runtime:** *Runtime → Change runtime type → T4 GPU.* Keep this tab open while it runs (free Colab stops when idle).

**Google Drive must contain:**
```
MyDrive/whistle_detector_5_24/
    code/whistle_code_5_24.zip            <- python mod_detector_5_24/make_colab_bundle_5_24.py
    encounter_191_background/*.flac       <- the 56 files (colab_download_backgrounds.py)
```

**Every session:** run cells 1–5. **First session only:** cell 6 (backgrounds, ~15–30 min) and cell 7 (timing test, ~10 min).
**Train:** cell 8. It saves its state to Drive every 250 steps; after a disconnect rerun cells 1–5 and then cell 8, and it
continues where it stopped. **When training has finished:** cell 9 (evaluation, ~15 min).

In [ ]:
# ===== 1. Settings =====
DRIVE      = "/content/drive/MyDrive/whistle_detector_5_24"   # project folder in Google Drive
VERSION    = "ridgenet-5-24"
CODE       = "/content/whistle_code"                        # unpacked code (fresh every session)
RAW        = "/content/encounter_191_audio"                 # 56 real files (links to Drive) + 307 name-only placeholders
LOCAL_DATA = "/content/data_5_24"                           # background banks and synthetic eval sets (fast local disk)
DRIVE_DATA = f"{DRIVE}/data_5_24"                           # their backup in Drive, restored every session

In [ ]:
# ===== 2. GPU, Python, Google Drive =====
import os, sys, torch
from google.colab import drive
assert sys.version_info >= (3, 11), f"needs Python >= 3.11, this runtime has {sys.version}"
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU"
print(f"GPU {torch.cuda.get_device_name(0)} | {os.cpu_count()} CPUs | Python {sys.version.split()[0]} | torch {torch.__version__}")
drive.mount("/content/drive")
for rel in ("code/whistle_code_5_24.zip", "encounter_191_background"):
    assert os.path.exists(f"{DRIVE}/{rel}"), f"missing in Drive: {DRIVE}/{rel}"
print("Drive OK")

In [ ]:
# ===== 3. Code: unpack a fresh copy every session =====
import importlib.util, json, shutil, subprocess, zipfile
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE}/code/whistle_code_5_24.zip") as z:
    z.extractall("/content")
if importlib.util.find_spec("soundfile") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile"], check=True)
if f"{CODE}/src" not in sys.path:
    sys.path.insert(0, f"{CODE}/src")
version = json.load(open(f"{CODE}/CODE_VERSION.json"))
print({k: v for k, v in version.items() if k != "files_sha256"})
if version["git_uncommitted_code_changes"]:
    print("NOTE: the bundle was made from uncommitted code; commit and rebuild the bundle for a run you will report")

In [ ]:
# ===== 4. Raw folder: the background picker lists all 363 file names, so the 307 files it never opens are empty placeholders =====
import csv
names = [r["filename"] for r in csv.DictReader(open(f"{CODE}/metadata/manifest_363.csv"))]
bg = {r["filename"]: int(r["file_size_bytes"]) for r in csv.DictReader(open(f"{CODE}/mod_detector_5_24/background_files.csv"))}
assert len(names) == 363 and len(bg) == 56 and set(bg) <= set(names)
shutil.rmtree(RAW, ignore_errors=True)
os.makedirs(RAW)
bad = []
for n in names:
    if n in bg:
        src = f"{DRIVE}/encounter_191_background/{n}"
        if not os.path.exists(src) or os.path.getsize(src) != bg[n]:
            bad.append(n)
            continue
        os.symlink(src, f"{RAW}/{n}")
    else:
        open(f"{RAW}/{n}", "wb").close()
assert not bad, f"missing or incomplete in Drive (re-run the download script): {bad}"
print(f"{RAW}: 56 background files linked from Drive, {363 - 56} placeholders")

In [ ]:
# ===== 5. Configuration and cached data =====
import whistlepipe.config as C
P_OVERRIDES = [f'paths.synth_raw_dir="{RAW}"', f'paths.data_dir="{LOCAL_DATA}"', f'paths.models_dir="{DRIVE}/models"']
P = C.load_pipeline_config(f"{CODE}/mod_detector_5_24/pipeline.toml", P_OVERRIDES)
D = C.load_detector_config(f"{CODE}/mod_detector_5_24/detector.toml")
k_lo, k_hi = C.band_bins(P)
print(f"band: bins {k_lo}-{k_hi} = {k_lo * 93.75:.1f}-{k_hi * 93.75:.1f} Hz | synthetic whistles {D['synth']['whistles']['f_range_hz']} Hz")
assert (k_lo, k_hi) == (54, 256)


def save_data():
    shutil.copytree(LOCAL_DATA, DRIVE_DATA, dirs_exist_ok=True)


if os.path.isdir(DRIVE_DATA):
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print("restored from Drive:", sorted(os.listdir(f"{LOCAL_DATA}/synthetic")))
else:
    print("no cached data yet: run cell 6")

In [ ]:
# ===== 6. First session: background banks + label-fix masks (~15-30 min; skipped when already built) =====
from whistlepipe.synth.backgrounds import bank_dir, build_all_banks, build_ridge_masks, pick_background_files
picks = pick_background_files(P, D)
assert {p.name for v in picks.values() for p in v} == set(bg), "picker chose files other than the 56 downloaded ones"
if (bank_dir(P) / "test.json").exists():
    print("background banks already built")
else:
    build_all_banks(P, D)
if (bank_dir(P) / "train_ridgemask.json").exists():
    meta = json.load(open(bank_dir(P) / "train_ridgemask.json"))
else:
    meta = build_ridge_masks(P, D, "train")
frac = 100 * meta["marked_fraction"]
print(f"label-fix mask marks {frac:.2f}% of background nodes (ridgenet-v2c, 2-24 kHz: 2.19%)")
if not 1.0 <= frac <= 4.0:
    print("WARNING: far from v2c; the bg_ridge_db threshold may need recalibrating for this band. Ask before training.")
save_data()

In [ ]:
# ===== 7. First session: timing test (300 steps, ~10 min; its model is discarded) =====
from whistlepipe.train import train_detector
TMP_MODELS = "/content/timing_models"
shutil.rmtree(TMP_MODELS, ignore_errors=True)
Pt = C.load_pipeline_config(f"{CODE}/mod_detector_5_24/pipeline.toml", P_OVERRIDES + [f'paths.models_dir="{TMP_MODELS}"'])
Dt = C.load_detector_config(f"{CODE}/mod_detector_5_24/detector.toml", ["train.steps=300"])
train_detector(Pt, Dt, "timing-test")
recs = [r for r in json.load(open(f"{TMP_MODELS}/detector/timing-test/train_log.json")) if "elapsed_s" in r]
sps = (recs[-1]["elapsed_s"] - recs[0]["elapsed_s"]) / (recs[-1]["step"] - recs[0]["step"])   # steps 100-300: after warm-up
print(f"\n{sps:.2f} s/step -> the 6000-step run needs about {6000 * sps / 3600:.1f} h of training")
save_data()   # keeps the synthetic validation set built here

### Cell 8: training
* Prints a line every 100 steps and a `VAL` line every 500. The best model so far is `models/detector/ridgenet-5-24/checkpoint.pt` in Drive.
* **Resume:** after a disconnect, rerun cells 1–5, then this cell. It prints `resuming after step N` and continues; the
  result is the same as an uninterrupted run (tested).

In [ ]:
# ===== 8. Train (resumable) =====
from whistlepipe.train import train_detector
mdir = f"{DRIVE}/models/detector/{VERSION}"
if os.path.exists(f"{mdir}/train_info.json"):
    print("training already finished:", json.load(open(f"{mdir}/train_info.json"))["best_val_node_f1"])
else:
    train_detector(P, D, VERSION, resume=True)
    save_data()

### Cell 9: evaluation (synthetic)
Calibrates the tracker threshold on the synthetic **validation** set, then evaluates the held-out **test** set once with
that threshold. Results go to `models/detector/ridgenet-5-24/` in Drive (`calibration.json`, `eval_val.json`, `eval_test.json`).

In [ ]:
# ===== 9. Evaluate: calibrate on validation, then test once =====
from whistlepipe.evaluate import evaluate_on_synthetic
mdir = f"{DRIVE}/models/detector/{VERSION}"
assert os.path.exists(f"{mdir}/train_info.json"), "training has not finished (cell 8)"
evaluate_on_synthetic(P, D, VERSION, "val", calibrate=True)
tau = json.load(open(f"{mdir}/calibration.json"))["tau"]
P_tau = C.load_pipeline_config(f"{CODE}/mod_detector_5_24/pipeline.toml", P_OVERRIDES + [f"decisions.s6.peak_threshold={tau}"])
res = evaluate_on_synthetic(P_tau, D, VERSION, "test")
m = res["track_level_at_config_tau"]
print(f"\nTEST (synthetic), tau={tau}: track recall {m['track_recall']:.3f}, "
      f"false tracks/min on clean negatives {m['false_tracks_per_min_surrogate_negative']:.2f}")
for b, v in m["recall_by_snr_db"].items():
    print(f"  SNR {b} dB: recall {v['recall']:.3f} ({v['recalled']}/{v['expected']})")
print(f"\nNext: set peak_threshold = {tau} in mod_detector_5_24/pipeline.toml [decisions.s6], and copy the Drive folder\n"
      f"  models/detector/{VERSION}/ into mod_detector_5_24/models/detector/{VERSION}/ in the project.")
save_data()